# LeetCode 3288 — Length of the Longest Increasing Path bằng Python

- Link: https://leetcode.com/problems/length-of-the-longest-increasing-path/
- Độ khó: **Hard**
- Chủ đề: **LIS 2 chiều** (tăng nghiêm ngặt cả `x` và `y`): sắp `x` tăng, **hòa `x` thì `y` giảm**, rồi LIS nghiêm ngặt trên `y`; đường đi **bắt buộc qua điểm `k`** ⇒ tách thành phần "trước" và "sau" `k`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxPathLength(self, coordinates: List[List[int]], k: int) -> int:
```

## 1. Mục tiêu học

1. Đường đi tăng: `x_i < x_{i+1}` **và** `y_i < y_{i+1}` (nghiêm ngặt cả hai).
2. Đường đi qua `P = coordinates[k]` ⇔ phần trước gồm các điểm **nhỏ hơn hẳn** `P` (cả hai tọa độ), phần sau gồm các điểm **lớn hơn hẳn** `P`. Hai phần độc lập ⇒ đáp án `= LIS(trước) + 1 + LIS(sau)`.
3. Kỹ thuật chuẩn LIS 2 chiều: sắp theo `x` tăng, cùng `x` thì `y` **giảm** ⇒ dãy con tăng nghiêm ngặt theo `y` không thể chứa hai điểm cùng `x`.
4. LIS `O(m log m)` bằng "patience sorting" (`bisect_left`).
5. Kiểm chứng bằng DP `O(n²)`.

## 2. Đề bài & ví dụ

Cho mảng điểm `coordinates` (phân biệt) và chỉ số `k`. Một **đường đi tăng** độ dài `m` là dãy điểm (thuộc tập) `(x_1, y_1), …, (x_m, y_m)` với `x_i < x_{i+1}` và `y_i < y_{i+1}`. Trả về độ dài lớn nhất của một đường đi tăng **chứa** `coordinates[k]`.

**Ví dụ 1:** `[[3,1],[2,2],[4,1],[0,0],[5,3]]`, `k = 1` → `3` (`(0,0), (2,2), (5,3)`).
**Ví dụ 2:** `[[2,1],[7,0],[5,6]]`, `k = 2` → `2`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `0 ≤ x, y ≤ 10^9`; các điểm phân biệt; `0 ≤ k < n`.

In [1]:
from typing import List
import bisect

examples = [([[3, 1], [2, 2], [4, 1], [0, 0], [5, 3]], 1, 3), ([[2, 1], [7, 0], [5, 6]], 2, 2)]
for pts, k, exp in examples:
    print(pts, 'k =', k, '-> kỳ vọng', exp)

[[3, 1], [2, 2], [4, 1], [0, 0], [5, 3]] k = 1 -> kỳ vọng 3
[[2, 1], [7, 0], [5, 6]] k = 2 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. LIS nghiêm ngặt bằng patience sorting

`tails[L]` = giá trị cuối nhỏ nhất của một dãy tăng độ dài `L + 1`. Với mỗi `v`: vị trí `bisect_left(tails, v)` — thay thế hoặc nối dài. `bisect_left` cho **tăng nghiêm ngặt** (bằng nhau không nối được).

In [2]:
def lis_strict(a):
    tails = []
    for v in a:
        i = bisect.bisect_left(tails, v)
        if i == len(tails):
            tails.append(v)
        else:
            tails[i] = v
    return len(tails)

print(lis_strict([3, 1, 2, 2, 5, 4, 6]), '(1, 2, 4, 6 hoặc 1, 2, 5, 6)')

4 (1, 2, 4, 6 hoặc 1, 2, 5, 6)


### 3.2. Sắp hòa `x` theo `y` giảm

Nếu sắp `(x, y)` tăng cả hai, hai điểm `(2, 1)`, `(2, 5)` đứng liền nhau và LIS trên `y` sẽ nối chúng — nhưng `x` bằng nhau không được phép. Sắp `y` **giảm** trong cùng `x` ⇒ trong dãy `y`, các điểm cùng `x` tạo thành đoạn **không tăng** ⇒ dãy tăng nghiêm ngặt lấy tối đa một điểm mỗi `x`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1, `P = (2, 2)`:

- Trước: điểm có `x < 2` và `y < 2`: `{(0, 0)}` ⇒ LIS = 1.
- Sau: điểm có `x > 2` và `y > 2`: `{(5, 3)}` ⇒ LIS = 1. (`(3, 1)`, `(4, 1)` có `y < 2` — không dùng được.)
- Đáp án `1 + 1 + 1 = 3` ✓.

## 5. Cách 1 — Brute Force: DP `O(n²)` trên thứ tự sắp

In [3]:
def brute(pts, k):
    order = sorted(range(len(pts)), key=lambda i: (pts[i][0], pts[i][1]))
    n = len(pts)
    end = [1] * n                                   # đường dài nhất kết thúc tại điểm
    for a in range(n):
        i = order[a]
        for b in range(a):
            j = order[b]
            if pts[j][0] < pts[i][0] and pts[j][1] < pts[i][1]:
                end[i] = max(end[i], end[j] + 1)
    start = [1] * n                                 # đường dài nhất bắt đầu tại điểm
    for a in range(n - 1, -1, -1):
        i = order[a]
        for b in range(a + 1, n):
            j = order[b]
            if pts[j][0] > pts[i][0] and pts[j][1] > pts[i][1]:
                start[i] = max(start[i], start[j] + 1)
    return end[k] + start[k] - 1

for pts, k, exp in examples:
    print(brute(pts, k), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2


**Độ phức tạp:** `O(n²)`.

## 6. Ý tưởng thuật toán / bước đột phá

```
(xk, yk) = coordinates[k]
before = [(x, y) với x < xk và y < yk];  after = [(x, y) với x > xk và y > yk]
mỗi nhóm: sắp theo (x tăng, y giảm), lấy dãy y, LIS nghiêm ngặt
return LIS(before) + 1 + LIS(after)
```

`O(n log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP `O(n²)` chỉ trên hai nhóm đã lọc

In [4]:
def filtered_quadratic(pts, k):
    xk, yk = pts[k]
    def longest(group):
        group = sorted(group)
        dp = [1] * len(group)
        for i in range(len(group)):
            for j in range(i):
                if group[j][0] < group[i][0] and group[j][1] < group[i][1]:
                    dp[i] = max(dp[i], dp[j] + 1)
        return max(dp, default=0)
    before = [(x, y) for x, y in pts if x < xk and y < yk]
    after = [(x, y) for x, y in pts if x > xk and y > yk]
    return longest(before) + 1 + longest(after)

for pts, k, exp in examples:
    print(filtered_quadratic(pts, k), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2


### 7.2. Cách 3 — Lọc + LIS patience sorting (lời giải chính)

In [5]:
def lis_2d(group):
    group.sort(key=lambda p: (p[0], -p[1]))
    return lis_strict([y for _, y in group])

def patience_solution(pts, k):
    xk, yk = pts[k]
    before = [(x, y) for x, y in pts if x < xk and y < yk]
    after = [(x, y) for x, y in pts if x > xk and y > yk]
    return lis_2d(before) + 1 + lis_2d(after)

for pts, k, exp in examples:
    print(patience_solution(pts, k), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2


### 7.3. Cách 4 — LIS bằng Fenwick tree (max) trên `y` đã nén

`best[y]` = đường dài nhất kết thúc ở một điểm có tung độ `y` (trong các điểm đã xét). Xử lý điểm theo `x` tăng; các điểm **cùng `x`** phải truy vấn trước rồi mới cập nhật (để không nối với nhau). Truy vấn `max` trên `y' < y`.

In [6]:
class FenwickMax:
    def __init__(self, n):
        self.n, self.t = n, [0] * (n + 1)
    def update(self, i, v):
        while i <= self.n:
            if self.t[i] < v:
                self.t[i] = v
            i += i & -i
    def query(self, i):              # max trên [1, i]
        r = 0
        while i > 0:
            r = max(r, self.t[i]); i -= i & -i
        return r

def fenwick_lis(group):
    if not group:
        return 0
    ys = sorted({y for _, y in group})
    rank = {y: i + 1 for i, y in enumerate(ys)}
    fw = FenwickMax(len(ys))
    group = sorted(group)
    best = 0; i = 0
    while i < len(group):
        j = i
        while j < len(group) and group[j][0] == group[i][0]:
            j += 1
        vals = [(rank[y], fw.query(rank[y] - 1) + 1) for _, y in group[i:j]]   # truy vấn cả nhóm trước
        for r, v in vals:
            fw.update(r, v); best = max(best, v)
        i = j
    return best

def fenwick_solution(pts, k):
    xk, yk = pts[k]
    return fenwick_lis([(x, y) for x, y in pts if x < xk and y < yk]) + 1 + \
        fenwick_lis([(x, y) for x, y in pts if x > xk and y > yk])

for pts, k, exp in examples:
    print(fenwick_solution(pts, k), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách 3 trên 8 điểm.

In [7]:
def trace(pts, k):
    xk, yk = pts[k]
    for name, grp in [('trước', [(x, y) for x, y in pts if x < xk and y < yk]),
                      ('sau', [(x, y) for x, y in pts if x > xk and y > yk])]:
        grp.sort(key=lambda p: (p[0], -p[1]))
        tails = []
        print(f'nhóm {name} (sắp x tăng, y giảm):', grp)
        for x, y in grp:
            i = bisect.bisect_left(tails, y)
            if i == len(tails):
                tails.append(y)
            else:
                tails[i] = y
            print(f'   thêm y = {y}: tails = {tails}')
    print('đáp án:', patience_solution(pts, k), '| DP O(n²):', brute(pts, k))

trace([[1, 1], [2, 3], [2, 2], [3, 2], [4, 4], [6, 5], [7, 7], [5, 8]], 4)

nhóm trước (sắp x tăng, y giảm): [(1, 1), (2, 3), (2, 2), (3, 2)]
   thêm y = 1: tails = [1]
   thêm y = 3: tails = [1, 3]
   thêm y = 2: tails = [1, 2]
   thêm y = 2: tails = [1, 2]
nhóm sau (sắp x tăng, y giảm): [(5, 8), (6, 5), (7, 7)]
   thêm y = 8: tails = [8]
   thêm y = 5: tails = [5]
   thêm y = 7: tails = [5, 7]
đáp án: 5 | DP O(n²): 5


## 9. Code nộp LeetCode

In [8]:
class Solution:
    def maxPathLength(self, coordinates: List[List[int]], k: int) -> int:
        xk, yk = coordinates[k]

        def lis(points):
            points.sort(key=lambda p: (p[0], -p[1]))
            tails = []
            for _, y in points:
                i = bisect.bisect_left(tails, y)
                if i == len(tails):
                    tails.append(y)
                else:
                    tails[i] = y
            return len(tails)

        before = [(x, y) for x, y in coordinates if x < xk and y < yk]
        after = [(x, y) for x, y in coordinates if x > xk and y > yk]
        return lis(before) + 1 + lis(after)

for pts, k, exp in examples:
    got = Solution().maxPathLength(pts, k)
    print(got, 'OK' if got == exp else 'SAI')

3 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Tách:** trong đường đi tăng chứa `P`, mọi điểm đứng trước `P` có cả hai tọa độ nhỏ hơn hẳn `P` (bắc cầu), mọi điểm sau lớn hơn hẳn. Ngược lại, ghép một đường tăng bất kỳ trong nhóm "trước", rồi `P`, rồi một đường tăng trong nhóm "sau" luôn được đường tăng (điểm cuối nhóm trước < `P` < điểm đầu nhóm sau) ⇒ hai phần **độc lập**, đáp án là tổng các cực đại cộng 1.
2. **LIS 2D:** sau khi sắp `(x tăng, y giảm)`, một dãy chỉ số tăng có `y` tăng nghiêm ngặt ⇒ `x` không giảm, và không thể có hai điểm cùng `x` (vì trong cùng `x`, `y` giảm) ⇒ `x` tăng nghiêm ngặt. Ngược lại mọi đường tăng là một dãy con như vậy ⇒ LIS trên `y` = đường tăng dài nhất.
3. **Patience sorting** với `bisect_left` tính đúng LIS nghiêm ngặt (bất biến `tails` tăng nghiêm ngặt).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DP toàn bộ | `O(n²)` | `O(n)` |
| 2. DP trên hai nhóm | `O(n²)` | `O(n)` |
| 3. Lọc + patience | `O(n log n)` | `O(n)` |
| 4. Lọc + Fenwick max | `O(n log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [9]:
impls = {'filtered_quadratic': filtered_quadratic, 'patience': patience_solution, 'fenwick': fenwick_solution,
         'submit': lambda p, k: Solution().maxPathLength(p, k)}
edge = [([[5, 5]], 0, 1), ([[1, 1], [1, 2], [1, 3]], 1, 1), ([[1, 1], [2, 1], [3, 1]], 0, 1),
        ([[0, 0], [1, 1], [2, 2], [3, 3]], 2, 4), ([[0, 3], [1, 2], [2, 1], [3, 0]], 1, 1),
        ([[1, 2], [2, 1], [3, 3]], 2, 2)]
for pts, k, exp in examples + edge:
    assert brute(pts, k) == exp, (pts, k, brute(pts, k))
    for name, f in impls.items():
        assert f([list(p) for p in pts], k) == exp, (name, pts, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với DP `O(n²)`

In [10]:
import random
rng = random.Random(3288)
cnt = 0
for _ in range(500):
    n = rng.randint(1, 30)
    V = rng.choice([4, 8, 50])
    pts = list({(rng.randint(0, V), rng.randint(0, V)) for _ in range(n)})
    pts = [list(p) for p in pts]
    k = rng.randrange(len(pts))
    exp = brute(pts, k)
    for name, f in impls.items():
        assert f([list(p) for p in pts], k) == exp, (name, pts, k)
    cnt += 1
print('Khớp DP trên', cnt, 'tập điểm ngẫu nhiên (nhiều điểm trùng x hoặc trùng y).')

Khớp DP trên 500 tập điểm ngẫu nhiên (nhiều điểm trùng x hoặc trùng y).


### 12.3. Ca lớn `n = 10^5`

In [11]:
import time
N = 10 ** 5
big = {
    'ngẫu nhiên': [[rng.randint(0, 10 ** 9), rng.randint(0, 10 ** 9)] for _ in range(N)],
    'đường chéo': [[i, i] for i in range(N)],
    'cùng x nhiều': [[rng.randint(0, 100), rng.randint(0, 10 ** 9)] for _ in range(N)],
}
for name, pts in big.items():
    seen = set(); uniq = []
    for p in pts:
        if tuple(p) not in seen:
            seen.add(tuple(p)); uniq.append(p)
    k = len(uniq) // 2
    t = time.time(); a = Solution().maxPathLength([p[:] for p in uniq], k); t1 = time.time() - t
    t = time.time(); b = fenwick_solution([p[:] for p in uniq], k); t2 = time.time() - t
    assert a == b
    print(f"{name:12s}: {a:6d} | nộp {t1:.2f}s | Fenwick {t2:.2f}s")
assert Solution().maxPathLength([[i, i] for i in range(N)], 123) == N
print('đường chéo: đáp án = N — đúng.')

ngẫu nhiên  :    611 | nộp 0.15s | Fenwick 0.29s


đường chéo  : 100000 | nộp 0.08s | Fenwick 0.34s


cùng x nhiều:    101 | nộp 0.14s | Fenwick 0.33s
đường chéo: đáp án = N — đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp hòa `x` theo `y` **tăng**

In [12]:
def wrong_tie_order(pts, k):
    xk, yk = pts[k]
    def lis(g):
        g = sorted(g)                                         # SAI: (x tăng, y tăng)
        return lis_strict([y for _, y in g])
    return lis([(x, y) for x, y in pts if x < xk and y < yk]) + 1 + lis([(x, y) for x, y in pts if x > xk and y > yk])

bad = [[0, 0], [1, 1], [1, 2], [5, 5]]
print('sai :', wrong_tie_order(bad, 3), '| đúng:', Solution().maxPathLength([p[:] for p in bad], 3))

sai : 4 | đúng: 3


**Vì sao sai:** `(1, 1)` và `(1, 2)` cùng `x` nhưng bị nối thành đường "tăng".

### Lỗi 2 — LIS không nghiêm ngặt (`bisect_right`)

In [13]:
def wrong_non_strict(pts, k):
    xk, yk = pts[k]
    def lis(g):
        g.sort(key=lambda p: (p[0], -p[1]))
        tails = []
        for _, y in g:
            i = bisect.bisect_right(tails, y)                 # SAI: cho phép y bằng nhau
            if i == len(tails):
                tails.append(y)
            else:
                tails[i] = y
        return len(tails)
    return lis([(x, y) for x, y in pts if x < xk and y < yk]) + 1 + lis([(x, y) for x, y in pts if x > xk and y > yk])

bad = [[0, 1], [1, 1], [2, 1], [5, 5]]
print('sai :', wrong_non_strict(bad, 3), '| đúng:', Solution().maxPathLength([p[:] for p in bad], 3))

sai : 4 | đúng: 2


**Vì sao sai:** đường đi cần `y` tăng **nghiêm ngặt**; ba điểm cùng `y = 1` không nối được.

### Lỗi 3 — Không ép đi qua điểm `k` (LIS toàn tập)

In [14]:
def wrong_global(pts, k):
    g = [tuple(p) for p in pts]
    return lis_2d(g)                                          # SAI

print('sai :', wrong_global(*examples[1][:2]), '| đúng:', Solution().maxPathLength(*examples[1][:2]))
bad = [[0, 0], [1, 1], [2, 2], [3, 0]]
print('sai :', wrong_global(bad, 3), '| đúng:', Solution().maxPathLength([p[:] for p in bad], 3))

sai : 2 | đúng: 2
sai : 3 | đúng: 1


**Vì sao sai:** đường dài nhất của cả tập có thể không chứa `coordinates[k]`; điểm `(3, 0)` chỉ có thể đứng một mình.

### Lỗi 4 — Lọc nhóm chỉ theo `x`

In [15]:
def wrong_filter_x_only(pts, k):
    xk, yk = pts[k]
    before = [(x, y) for x, y in pts if x < xk]               # SAI: thiếu điều kiện y < yk
    after = [(x, y) for x, y in pts if x > xk]
    return lis_2d(before) + 1 + lis_2d(after)

print('sai :', wrong_filter_x_only(*examples[0][:2]), '| đúng:', Solution().maxPathLength(*examples[0][:2]))

sai : 4 | đúng: 3


**Vì sao sai:** `(3, 1)` có `x > 2` nhưng `y < 2` — không thể đi sau `(2, 2)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Chuỗi tăng nghiêm ngặt theo hai chiều" (búp bê Nga LC 354, xếp hộp) ⇒ sắp chiều 1 tăng, hòa thì chiều 2 **giảm**, rồi LIS chiều 2.
- "Đường đi bắt buộc qua phần tử `k`" ⇒ tách trước/sau, hai bài con độc lập.
- LIS nghiêm ngặt dùng `bisect_left`, không nghiêm ngặt dùng `bisect_right`.

## 15. Biến thể của bài

1. **LC 354 Russian Doll Envelopes** — đúng LIS 2D không có ràng buộc qua `k`.
2. Đường đi qua **tập** điểm bắt buộc (đã sắp được) ⇒ ghép nhiều đoạn.
3. Đếm **số** đường đi tăng dài nhất qua `k` ⇒ Fenwick lưu `(độ dài, số cách)`.
4. Chỉ cần `x` **không giảm**, `y` tăng nghiêm ngặt ⇒ sắp hòa `x` theo `y` **tăng**.

In [16]:
# Biến thể 1 và 4
def russian_doll(envs):
    return lis_2d([tuple(e) for e in envs])

def x_nondecreasing(pts):
    g = sorted(tuple(p) for p in pts)                         # hòa x: y tăng -> được nối
    return lis_strict([y for _, y in g])

assert russian_doll([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3
rng = random.Random(15)
for _ in range(300):
    pts = list({(rng.randint(0, 5), rng.randint(0, 5)) for _ in range(rng.randint(1, 12))})
    order = sorted(pts)
    dp = [1] * len(order)
    for i in range(len(order)):
        for j in range(i):
            if order[j][0] <= order[i][0] and order[j][1] < order[i][1]:
                dp[i] = max(dp[i], dp[j] + 1)
    assert x_nondecreasing(pts) == max(dp)
print('LC 354 ví dụ -> 3; biến thể x không giảm khớp DP.')

LC 354 ví dụ -> 3; biến thể x không giảm khớp DP.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | DP toàn bộ | `O(n²)` | Oracle |
| 2 | DP hai nhóm | `O(n²)` | Thấy rõ tách bài |
| 3 | Lọc + patience | `O(n log n)` | **Nộp bài** |
| 4 | Lọc + Fenwick max | `O(n log n)` | Mở rộng được (đếm số cách) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[0,0],[1,1],[2,2]]`, `k = 0`.

**Bài 2 (tính tay):** `[[1,5],[2,4],[3,3],[4,2],[5,1]]`, `k = 2`.

**Bài 3 (điền TODO):** trả về luôn **một đường đi** dài nhất qua `k`.

**Bài 4 (tự viết lại):** với **mọi** `k`, độ dài lớn nhất (`O(n log n)` tổng).

**Bài 5 (mini challenge):** biến thể 3 — đếm số đường đi tăng dài nhất qua `k`.

In [17]:
def one_path_todo(pts, k):
    # TODO
    raise NotImplementedError

def all_k_todo(pts):
    # TODO
    raise NotImplementedError

def count_longest_todo(pts, k):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đường chéo. (2) Mọi điểm lớn hơn `(0,0)`. (3) `3`.

**Bài 2.** (1) Đường chéo phụ: không cặp nào tăng cả hai. (2) Nhóm trước/sau rỗng. (3) `1`.

**Bài 3.** (1) Patience có lưu "cha". (2) Dựng ngược. (3) Ghép trước + `P` + sau.

**Bài 4.** (1) `end[i]` = LIS kết thúc tại `i` (Fenwick theo x tăng), `start[i]` = LIS bắt đầu tại `i` (Fenwick theo x giảm). (2) `end[i] + start[i] − 1`. (3) So với DP.

**Bài 5.** (1) Fenwick lưu `(độ dài max, số cách)` với phép gộp. (2) Làm cho nhóm trước (kết thúc) và nhóm sau (bắt đầu). (3) Nhân số cách của hai phía.

## 19. Đáp án tham khảo cho bài tập

In [18]:
# Bài 1, 2, 3
assert Solution().maxPathLength([[0, 0], [1, 1], [2, 2]], 0) == 3
assert Solution().maxPathLength([[1, 5], [2, 4], [3, 3], [4, 2], [5, 1]], 2) == 1

def lis_with_path(group):
    group = sorted(group, key=lambda p: (p[0], -p[1]))
    tails, tail_idx, parent = [], [], [-1] * len(group)
    for i, (_, y) in enumerate(group):
        j = bisect.bisect_left(tails, y)
        parent[i] = tail_idx[j - 1] if j else -1
        if j == len(tails):
            tails.append(y); tail_idx.append(i)
        else:
            tails[j] = y; tail_idx[j] = i
    path, i = [], tail_idx[-1] if tail_idx else -1
    while i >= 0:
        path.append(group[i]); i = parent[i]
    return path[::-1]

def one_path(pts, k):
    xk, yk = pts[k]
    return lis_with_path([(x, y) for x, y in pts if x < xk and y < yk]) + [(xk, yk)] + \
        lis_with_path([(x, y) for x, y in pts if x > xk and y > yk])

rng = random.Random(3)
for _ in range(300):
    pts = [list(p) for p in {(rng.randint(0, 8), rng.randint(0, 8)) for _ in range(rng.randint(1, 15))}]
    k = rng.randrange(len(pts))
    path = one_path(pts, k)
    assert len(path) == brute(pts, k) and tuple(pts[k]) in path
    assert all(a[0] < b[0] and a[1] < b[1] for a, b in zip(path, path[1:]))
    assert all(list(p) in pts for p in path)
print('Bài 1: 3 | Bài 2: 1 | Bài 3: đường đi dựng được hợp lệ và dài nhất.')

Bài 1: 3 | Bài 2: 1 | Bài 3: đường đi dựng được hợp lệ và dài nhất.


In [19]:
# Bài 4, 5
def all_k(pts):
    n = len(pts)
    ys = sorted({y for _, y in pts}); rank = {y: i + 1 for i, y in enumerate(ys)}; m = len(ys)
    order = sorted(range(n), key=lambda i: pts[i][0])
    end = [0] * n; fw = FenwickMax(m); i = 0
    while i < n:
        j = i
        while j < n and pts[order[j]][0] == pts[order[i]][0]:
            j += 1
        for t in order[i:j]:
            end[t] = fw.query(rank[pts[t][1]] - 1) + 1
        for t in order[i:j]:
            fw.update(rank[pts[t][1]], end[t])
        i = j
    start = [0] * n; fw = FenwickMax(m); i = n - 1
    while i >= 0:
        j = i
        while j >= 0 and pts[order[j]][0] == pts[order[i]][0]:
            j -= 1
        for t in order[j + 1:i + 1]:
            start[t] = fw.query(m - rank[pts[t][1]]) + 1       # đảo trục y: y' > y
        for t in order[j + 1:i + 1]:
            fw.update(m - rank[pts[t][1]] + 1, start[t])
        i = j
    return [end[t] + start[t] - 1 for t in range(n)]

def count_longest(pts, k):
    xk, yk = pts[k]
    def cnt_best(group):
        if not group:
            return 0, 1
        group = sorted(group)
        dp = [(1, 1)] * len(group)
        for i in range(len(group)):
            L, c = 1, 1
            for j in range(i):
                if group[j][0] < group[i][0] and group[j][1] < group[i][1]:
                    if dp[j][0] + 1 > L:
                        L, c = dp[j][0] + 1, dp[j][1]
                    elif dp[j][0] + 1 == L:
                        c += dp[j][1]
            dp[i] = (L, c)
        best = max(L for L, _ in dp)
        return best, sum(c for L, c in dp if L == best)
    _, a = cnt_best([(x, y) for x, y in pts if x < xk and y < yk])
    _, b = cnt_best([(x, y) for x, y in pts if x > xk and y > yk])
    return a * b

rng = random.Random(4)
for _ in range(200):
    pts = [list(p) for p in {(rng.randint(0, 6), rng.randint(0, 6)) for _ in range(rng.randint(1, 12))}]
    assert all_k(pts) == [brute(pts, k) for k in range(len(pts))]
print('Bài 4 khớp DP với mọi k. Bài 5 ví dụ: số đường dài nhất qua (2,2) trong lưới 5×5 =',
      count_longest([[i, j] for i in range(5) for j in range(5)], 12))

Bài 4 khớp DP với mọi k. Bài 5 ví dụ: số đường dài nhất qua (2,2) trong lưới 5×5 = 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
before = [p với x < xk và y < yk];  after = [p với x > xk và y > yk]
lis(g): sort(x tăng, y giảm); patience trên y với bisect_left
return lis(before) + 1 + lis(after)
```

**Checklist:**

- [ ] Lọc theo **cả hai** tọa độ?
- [ ] Hòa `x` ⇒ `y` giảm?
- [ ] `bisect_left` (nghiêm ngặt)?
- [ ] Cộng 1 cho chính điểm `k`?
- [ ] Nhóm rỗng ⇒ LIS = 0?